# Reconciliación de la cosecha OAI

Compara manifiesto y registros activos, y prueba la consolidación de la cosecha masiva con las recuperaciones individuales.


In [ ]:
import inspect
import os
import time
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

import certifi
import pandas as pd
import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning


In [ ]:
def oai_find_missing_record_identifiers(
    manifest: pd.DataFrame,
    records: pd.DataFrame,
) -> pd.DataFrame:
    """Return active manifest entries that are absent from harvested records."""
    active = manifest.loc[~manifest["is_deleted"].fillna(False)].copy()
    recovered_ids = records["record_id"].dropna().unique()
    return (
        active.loc[~active["record_id"].isin(recovered_ids)]
        .drop_duplicates(subset=["record_id"], keep="last")
        .reset_index(drop=True)
    )


In [ ]:
def oai_merge_harvested_records(
    records: pd.DataFrame,
    recovered_records: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Consolidate bulk and GetRecord results into the latest raw snapshot."""
    consolidated = (
        pd.concat([records, recovered_records], ignore_index=True)
        .drop_duplicates(subset=["record_id"], keep="last")
        .reset_index(drop=True)
    )
    return consolidated, consolidated.head(100)


In [ ]:
df_identifiers = catalog.load("raw/oai/identifiers#parquet")
df_records = catalog.load("raw/oai/records#parquet")
df_recovered = catalog.load("raw/oai/records_recovered#parquet")


In [ ]:
df_missing = oai_find_missing_record_identifiers(df_identifiers, df_records)
df_consolidated, df_consolidated_preview = oai_merge_harvested_records(df_records, df_recovered)
assert not df_consolidated["record_id"].duplicated().any()
{"missing_against_current_snapshot": len(df_missing), "consolidated_records": len(df_consolidated)}


In [ ]:
display(df_missing.head(20))
display(df_consolidated_preview)
